# Trabajo Practico Especial. Teoría de la Información 2024.
**Alumnos:** Villa Eliseo, Román Nahuel.

Se dispone de tres señales (S1,S2 y S3) correspondientes a los valores diarios de temperatura promedio de tres ciudades diferentes (Buenos Aires, Bogotá y Vancouver) durante cierto periodo de tiempo.



---


## Ejercicio 1.
Se requiere hacer diferentes análisis estadísticos a partir de estos valores, por lo que se solicita obtener:


*   Media y desvío de cada una de las señales. Analizar y comparar.
*   Factor de correlación cruzada para cada par de señales. Analizar y comparar.

---
*Cofiguración Inicial*:

1. **Importar** (examinar) los archivos de las señales **manualmente**.
 * Examinar y seleccionar todos los archivos de las carpetas "señales"

In [1]:
from collections import Counter,defaultdict
import numpy as np
import os
import re
import matplotlib.pyplot as plt
from pprint import pprint

In [2]:
from google.colab import files
uploaded = files.upload()

Saving S1_buenosAires.csv to S1_buenosAires.csv
Saving S2_bogota.csv to S2_bogota.csv
Saving S3_vancouver.csv to S3_vancouver.csv
Saving S4_buenosAiresR.csv to S4_buenosAiresR.csv


---
Una vez cargado los archivos, comenzamos a trabajar.

---

Comenzamos con el punto **(a)** del enunciado: Media y desvío de cada una de las señales.

Para poder resolver este inciso, seguimos los siguientes pasos para cada señal:

1.   Guardamos la temperaturas en un arreglo.
2.   Calculamos la distribución de Probabilidades.
3.   Calculamos la media.
4.   Calculamos la media cuadrática y el desvio.




In [3]:
def cargar_distr_prob(senal):
	total = len(senal)
	distr_actual = {}
	for simbolo in senal:
		if simbolo in distr_actual:
			distr_actual[simbolo] += 1 / total
		else:
			distr_actual[simbolo] = 1 / total
	return distr_actual

In [4]:
def calcular_media(distribucion):
	media = 0
	for simbolo in distribucion:
		media += simbolo * distribucion[simbolo]
	return media

In [5]:
def calcular_media_cuadratica(distribucion):
	media_cuadratica = 0
	for simbolo in distribucion:
		media_cuadratica += simbolo * simbolo * distribucion[simbolo]
	return media_cuadratica

In [6]:
def calcular_desvio(media,media_cuadratica):
	return (np.sqrt(media_cuadratica - (media*media))) # media_cuadratica - (media*media) es la varianza

In [7]:
s1_buenos_aires = np.loadtxt('/content/S1_buenosAires.csv',dtype=float,skiprows=0)
s2_bogota = np.loadtxt('/content/S2_bogota.csv',dtype=float,skiprows=0)
s3_vancouver = np.loadtxt('/content/S3_vancouver.csv',dtype=float,skiprows=0)
s4_buenos_aires_r = np.loadtxt('/content/S4_buenosAiresR.csv',dtype=float,skiprows=0)

print("--- Muestra de las 10 primeras temperaturas registradas ---")
print("Buenos Aires: ", s1_buenos_aires[:10])
print("Bogota: ",s2_bogota[:10])
print("Vancouver: ",s3_vancouver[:10])
#print("Buenos Aires luego del canal: ",s4_buenos_aires_r[:10])

--- Muestra de las 10 primeras temperaturas registradas ---
Buenos Aires:  [28. 23. 23. 25. 26. 21. 21. 24. 19. 16.]
Bogota:  [12. 14. 10. 11. 11. 11. 11. 13. 13. 10.]
Vancouver:  [ 0. -1. -1. -1. -1. -1.  0.  4.  6.  7.]


In [8]:
s1_distr_prob = cargar_distr_prob(s1_buenos_aires)
s2_distr_prob = cargar_distr_prob(s2_bogota)
s3_distr_prob = cargar_distr_prob(s3_vancouver)

s1_media = calcular_media(s1_distr_prob)
s2_media = calcular_media(s2_distr_prob)
s3_media = calcular_media(s3_distr_prob)

s1_media_cuadratica = calcular_media_cuadratica(s1_distr_prob)
s2_media_cuadratica = calcular_media_cuadratica(s2_distr_prob)
s3_media_cuadratica = calcular_media_cuadratica(s3_distr_prob)

s1_desvio = calcular_desvio(s1_media,s1_media_cuadratica)
s2_desvio = calcular_desvio(s2_media,s2_media_cuadratica)
s3_desvio = calcular_desvio(s3_media,s3_media_cuadratica)

print("--- Valor medio de temperatura ---")
print("<Buenos Aires> = ",s1_media)
print("<Bogota> = ",s2_media)
print("<Vancouver> = ",s3_media)
print("")
print("--- Desvio estándar ---")
print("Buenos Aires σ = ",s1_desvio)
print("Bogota σ = ",s2_desvio)
print("Vancouver σ = ",s3_desvio)

--- Valor medio de temperatura ---
<Buenos Aires> =  16.523342939481346
<Bogota> =  12.895677233429577
<Vancouver> =  10.068011527377573

--- Desvio estándar ---
Buenos Aires σ =  5.98291537904587
Bogota σ =  1.0631684415051383
Vancouver σ =  5.624666478220835




---

con el punto **(b)** del enunciado: Factor de correlación cruzada para cada par de señales.


In [9]:
def covarianza(senales1, senales2, media1, media2):
  len1 = len(senales1)
  len2 = len(senales2)
  len_min = min(len1, len2)

  calc_aux = 0

  senales1 = np.array(senales1[:len_min])
  senales2 = np.array(senales2[:len_min])

  calc_aux = np.sum((senales1 - media1) * (senales2 - media2))
  cov = calc_aux / len_min

  # https://sf.ezoiccdn.com/ezoimgfmt/www.probabilidadyestadistica.net/wp-content/uploads/2021/12/covarianza.png?ezimgfmt=ng:webp/ngcb1
  # https://www.probabilidadyestadistica.net/covarianza/

  return cov # cov(a,b) = la sumatoria de (xi-xmedia)*(yi-ymedia) / cant_simbolos


In [10]:
def coeficiente_correlacion_lineal(cov, desviacionT1, desviacionT2):
  ccl = (cov)/(desviacionT1*desviacionT2)

  if(ccl == 1):
    print("Existe una correlación positiva perfecta")
  elif(ccl >= 0.1):
    print("Existe una correlación positiva")
  elif(ccl == -1):
    print("Existe una correlación negativa perfecta")
  elif(ccl <= -0.1):
    print("Existe una correlación negativa")
  else:
    print("Hay poca o nula relacion lineal")
  return ccl


In [11]:
print("--- Factor de correlación cruzada entre par de señales ---")
print("• Relacion de temperaturas entre Buenos Aires y Bogota")
cov_bsas_bog = covarianza(s1_buenos_aires,s2_bogota, s1_media, s2_media)
r_bsas_bog = coeficiente_correlacion_lineal(cov_bsas_bog,s1_desvio,s2_desvio)
print("r(BsAs, Bog) = ",r_bsas_bog)
print("")

print("• Relacion de temperaturas entre Buenos Aires y Vancouver")
cov_bsas_vanc = covarianza(s1_buenos_aires,s3_vancouver, s1_media, s3_media)
r_bsas_vanc = coeficiente_correlacion_lineal(cov_bsas_vanc,s1_desvio,s3_desvio)
print("r(BAs, Vanc) = ",r_bsas_vanc)
print("")

print("• Relacion de temperaturas entre Bogota y Vancouver")
cov_bog_vanc = covarianza(s2_bogota,s3_vancouver, s2_media, s3_media)
r_bog_vanc = coeficiente_correlacion_lineal(cov_bog_vanc,s2_desvio,s3_desvio)
print("r(Bog, Vanc) = ",r_bog_vanc)
print("")

#https://www.estrategiasdeinversion.com/herramientas/diccionario/analisis-tecnico/coeficiente-de-correlacion-en-finanzas-como-construir-t-1245

--- Factor de correlación cruzada entre par de señales ---
• Relacion de temperaturas entre Buenos Aires y Bogota
Hay poca o nula relacion lineal
r(BsAs, Bog) =  -0.05344066946234848

• Relacion de temperaturas entre Buenos Aires y Vancouver
Existe una correlación negativa
r(BAs, Vanc) =  -0.7306368985877647

• Relacion de temperaturas entre Bogota y Vancouver
Hay poca o nula relacion lineal
r(Bog, Vanc) =  0.03933016658064952





---


## Ejercicio 2
A partir de cada una de las señales Si se desea crear una nueva señal Ti que indique si la temperatura **t** registrada cada día es **Baja, Alta o Moderada**, considerando los siguientes rangos:

* (B)aja: si t < 10
* (M)oderada: si 10 <= t < 20
* (A)lta: si t >= 20

In [12]:
def clasificar_temperatura(temperaturas):
    clasificacion = []
    for t in temperaturas:
        if t < 10:
            clasificacion.append('B')
        elif t < 20:
            clasificacion.append('M')
        else:
            clasificacion.append('A')
    return clasificacion

# Clasificar temperaturas
clasificacion_ba = clasificar_temperatura(s1_buenos_aires)
clasificacion_bo = clasificar_temperatura(s2_bogota)
clasificacion_va = clasificar_temperatura(s3_vancouver)

print(clasificacion_ba[:10])
print(clasificacion_bo[:10])
print(clasificacion_va[:10])

distribucion_ba = cargar_distr_prob(clasificacion_ba)
distribucion_bo = cargar_distr_prob(clasificacion_bo)
distribucion_va = cargar_distr_prob(clasificacion_va)

print(distribucion_ba)
print(distribucion_bo)
print(distribucion_va)

['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'M', 'M']
['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M']
['B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B']
{'A': 0.3466858789625415, 'M': 0.5103746397694567, 'B': 0.14293948126801342}
{'M': 0.9963976945243213, 'B': 0.0036023054755043218}
{'B': 0.491642651296838, 'M': 0.4805475504322845, 'A': 0.027809798270893332}


*Modelar cada señal Ti como una fuente Markoviana, obteniendo su matriz de pasaje*

In [13]:
def calcular_matriz_pasaje(clasificacion):
    # Counter: https://www.geeksforgeeks.org/counters-in-python-set-1/
    # defaultdict: https://www.geeksforgeeks.org/defaultdict-in-python/
    # zip: Por ejemplo, si clasificacion es [B, M, A, B]
                # zip(clasificacion[:-1], clasificacion[1:]) va a generar [(B, M), (M, A), (A, B)]

    # Inicializamos un diccionario para contar transiciones.
    transiciones = defaultdict(Counter)

    # Contamos las transiciones entre estados consecutivos
    for (estado_actual, estado_siguiente) in zip(clasificacion[:-1], clasificacion[1:]):
        transiciones[estado_actual][estado_siguiente] += 1

    # Normalizamos las transiciones para obtener probabilidades, la matriz de pasaje es un diccionario de diccionarios que tienen valor por defecto 0.0 si la clave no existe.
    matriz_pasaje = defaultdict(lambda: defaultdict(float))
    for estado in transiciones:
        total_transiciones = sum(transiciones[estado].values())
        for siguiente in transiciones[estado]:
            matriz_pasaje[estado][siguiente] = transiciones[estado][siguiente] / total_transiciones

    return matriz_pasaje

matriz_pasaje_ba = calcular_matriz_pasaje(clasificacion_ba)
matriz_pasaje_bo = calcular_matriz_pasaje(clasificacion_bo)
matriz_pasaje_va = calcular_matriz_pasaje(clasificacion_va)

print("--- Matriz de pasaje ---")
print("Buenos Aires")
pprint(dict(matriz_pasaje_ba))
print("")
print("Bogota")
pprint(dict(matriz_pasaje_bo))
print("")
print("Vancouver")
pprint(dict(matriz_pasaje_va))


--- Matriz de pasaje ---
Buenos Aires
{'A': defaultdict(<class 'float'>,
                  {'A': 0.8187110187110187,
                   'M': 0.1812889812889813}),
 'B': defaultdict(<class 'float'>,
                  {'B': 0.6905241935483871,
                   'M': 0.3094758064516129}),
 'M': defaultdict(<class 'float'>,
                  {'A': 0.12309429700734048,
                   'B': 0.08667419536984754,
                   'M': 0.790231507622812})}

Bogota
{'B': defaultdict(<class 'float'>, {'M': 0.84, 'B': 0.16}),
 'M': defaultdict(<class 'float'>,
                  {'B': 0.00303731559155337,
                   'M': 0.9969626844084466})}

Vancouver
{'A': defaultdict(<class 'float'>,
                  {'A': 0.5958549222797928,
                   'M': 0.40414507772020725}),
 'B': defaultdict(<class 'float'>,
                  {'B': 0.9325710935209616,
                   'M': 0.0674289064790384}),
 'M': defaultdict(<class 'float'>,
                  {'A': 0.023388305847076463,
     


---

a) Calcular Entropia para la fuente **con** y **sin** memoria.

In [14]:
def vector_estacionario(matriz_pasaje):
  minimo = 0.00001
  error = True
  vector_ant = {}
  vector_actual = {}
  resta = 0
  i=1
  for s in matriz_pasaje:
    vector_ant[s]=i # A: 1 , B: 0, C: 0
    i=0
    vector_actual[s]=i # A: 0 B :0 C : 0

  #i=0

  while(error):
    #print("i = ",i)
    #print("--------------")
    for x in matriz_pasaje:
     # print("X = ",x)
      for y in matriz_pasaje[x]:
      #  print("Y = ",y)
       # print("(",vector_ant[y],"*",matriz_pasaje[y][x],")+",vector_actual[x] )
        vector_actual[x]=(vector_ant[y]*matriz_pasaje[y][x])+vector_actual[x]


    for s in vector_actual:
      resta = abs(vector_actual[s]-vector_ant[s])
      vector_ant[s] = vector_actual[s]
      vector_actual[s]=0
      if(resta<=minimo and resta!=0):
        error = False
      else:
        error = True
    #i+=1
    #print(vector_ant)
  return vector_ant

In [15]:
def calcular_entropia(probabilidades):
    entropia = 0
    for simbolo in probabilidades: # P(simbolo) donde simbolo = {A,M,B}
      entropia -= probabilidades[simbolo] * np.log2(probabilidades[simbolo])
    return entropia

In [16]:
def calcular_entropia_condicional(vect_estacionario,matriz_pasaje):
    entropia_condicional = 0
    for estado in matriz_pasaje:
        for siguiente in matriz_pasaje[estado]:
            probabilidad = matriz_pasaje[estado][siguiente] # P(j/i)
            entropia_condicional -= vect_estacionario[estado]* probabilidad * np.log2(probabilidad) # += P(j/i) * log2(P(j/i))
    return entropia_condicional

In [17]:
# Vector estacionario
vect_est_ba = vector_estacionario(matriz_pasaje_ba)
vect_est_bo = vector_estacionario(matriz_pasaje_bo)
vect_est_va = vector_estacionario(matriz_pasaje_va)

# Entropias sin memoria
entropia_ba_sin_mem = calcular_entropia(distribucion_ba)
entropia_bo_sin_mem = calcular_entropia(distribucion_bo)
entropia_va_sin_mem = calcular_entropia(distribucion_va)

# Entropias condicionales
entropia_ba_cond = calcular_entropia_condicional(vect_est_ba,matriz_pasaje_ba)
entropia_bo_cond = calcular_entropia_condicional(vect_est_bo,matriz_pasaje_bo)
entropia_va_cond = calcular_entropia_condicional(vect_est_va,matriz_pasaje_va)

print("--- Vectores estacionarios V* ---")
print("V* Buenos Aires", vect_est_ba)
print("V* Bogota", vect_est_bo)
print("V* Vancouver", vect_est_va)
print("")
print("--- Entropia (H1) ---")
print("H Buenos Aires", entropia_ba_sin_mem)
print("H Bogota", entropia_bo_sin_mem)
print("H Vancouver", entropia_va_sin_mem)
print("")
print("--- Entropia Condicional (Hcond) ---")
print("Hcond Buenos Aires", entropia_ba_cond)
print("Hcond Bogota", entropia_bo_cond)
print("Hcond Vancouver", entropia_va_cond)


--- Vectores estacionarios V* ---
V* Buenos Aires {'A': 0.3466455867144514, 'M': 0.5104237091426497, 'B': 0.14293070414289977}
V* Bogota {'M': 0.9963975186470911, 'B': 0.0036024813529086912}
V* Vancouver {'B': 0.49236327305203614, 'M': 0.47988552702981924, 'A': 0.02775119991814398}

--- Entropia (H1) ---
H Buenos Aires 1.4262557016176847
H Bogota 0.03442707910187602
H Vancouver 1.155385143935258

--- Entropia Condicional (Hcond) ---
Hcond Buenos Aires 0.847270831113731
Hcond Bogota 0.03195411078714157
Hcond Vancouver 0.45180182876300307


**b)** Generar un conjunto de códigos mediante Huffman para los símbolos de cada señal Ti
original y para la extensión de cada señal a orden 2 (Ti^2).

Calcular la **longitud promedio** de cada codificación y **analizar según el 1° teorema de Shannon.**


In [ ]:
def extension_fuente(vector_estacionario, matriz_pasaje, orden=1):
  fuente = []

  if(orden==1):
    for s1 in vector_estacionario:
      simbolo = [vector_estacionario[s1],s1]
      fuente.append(simbolo)
  else:
    for s1 in vector_estacionario:
      for s2 in vector_estacionario:
          aux = [0,""]
          prob = vector_estacionario[s1]*matriz_pasaje[s1][s2]
          if prob != 0.0:
            aux[0]=prob
            aux[1]=(s1+s2)
            fuente.append(aux)

 # fuente_filtrada = [sublista for sublista in fuente if sublista[0] != 0.0]

  return fuente

In [ ]:
def huffman_codificacion(fuente_extendida):
  #print("---------")
  while(fuente_extendida[0][0]<0.99999):
    fuente_extendida.sort(key=lambda x: x[0])
    #print("f_ext: ",fuente_extendida)

    prob = fuente_extendida[0][0]+fuente_extendida[1][0]

    nueva_cod = [prob]

    for i in range(1,len(fuente_extendida[0])):
      nueva_cod.append("0"+fuente_extendida[0][i])

    for i in range(1,len(fuente_extendida[1])):
      nueva_cod.append("1"+fuente_extendida[1][i])

    fuente_extendida = fuente_extendida[2:]

    fuente_extendida.append(nueva_cod)

  #print("---------")
  #print("")

  codificacion = {}
  #print(fuente_extendida)
  for i in range(1,len(fuente_extendida[0])):
    cod = re.match(r"(\d+)(\D+)",fuente_extendida[0][i])
    if cod:
      codificacion[cod.group(2)]=cod.group(1)

  return codificacion

In [ ]:
def longitud_codificacion(codificacion,fuente):
  longitud_cod = 0
  for tupla in fuente:
    longitud_cod += tupla[0]*len(codificacion[tupla[1]])
  return longitud_cod

In [ ]:
def teorema_shannon(entropia, longitud, entropia_condicional=0.0, orden=1):
  primer_condicion = (entropia+((orden-1)*entropia_condicional))/orden
  parametro = longitud/orden
  segunda_condicion = (entropia+((orden-1)*entropia_condicional)+1)/orden

  if((primer_condicion <= parametro < segunda_condicion)):
    return "Cumple el teorema de Shannon"
  else:
    return "No cumple el teorema de Shannon. Hay un error en los calculos"

In [ ]:
def tasa_compresion(clasificacion, codificacion, orden, tam_datos_originales):
  tam_datos_comprimido = 0
  cant_simbolos = len(clasificacion)
  i= 0
  while(cant_simbolos>i):
    if (orden==1):
      simbolo = clasificacion[i]
    else:
      simbolo = clasificacion[i]+clasificacion[i+1]
    tam_datos_comprimido += len(codificacion[simbolo])
    i += orden

  return (tam_datos_originales/tam_datos_comprimido),tam_datos_comprimido

In [ ]:
f_extendida_ba = extension_fuente(vect_est_ba,matriz_pasaje_ba, 2)
f_extendida_bo = extension_fuente(vect_est_bo,matriz_pasaje_bo, 2)
f_extendida_va = extension_fuente(vect_est_va,matriz_pasaje_va, 2)

f_sin_extender_ba = extension_fuente(vect_est_ba,matriz_pasaje_ba,1)
f_sin_extender_bo = extension_fuente(vect_est_bo,matriz_pasaje_bo,1)
f_sin_extender_va = extension_fuente(vect_est_va,matriz_pasaje_va,1)

huffman_f_extendida_ba = huffman_codificacion(f_extendida_ba.copy())
huffman_f_extendida_bo = huffman_codificacion(f_extendida_bo.copy())
huffman_f_extendida_va = huffman_codificacion(f_extendida_va.copy())

huffman_f_sin_extender_ba = huffman_codificacion(f_sin_extender_ba.copy())
huffman_f_sin_extender_bo = huffman_codificacion(f_sin_extender_bo.copy())
huffman_f_sin_extender_va = huffman_codificacion(f_sin_extender_va.copy())

long_cod_f_extendida_ba = longitud_codificacion(huffman_f_extendida_ba,f_extendida_ba)
long_cod_f_extendida_bo = longitud_codificacion(huffman_f_extendida_bo,f_extendida_bo)
long_cod_f_extendida_va = longitud_codificacion(huffman_f_extendida_va,f_extendida_va)

long_cod_f_sin_extender_ba = longitud_codificacion(huffman_f_sin_extender_ba,f_sin_extender_ba)
long_cod_f_sin_extender_bo = longitud_codificacion(huffman_f_sin_extender_bo,f_sin_extender_bo)
long_cod_f_sin_extender_va = longitud_codificacion(huffman_f_sin_extender_va,f_sin_extender_va)

tasa_compresion_f_sin_extender_ba = tasa_compresion(clasificacion_ba,huffman_f_sin_extender_ba,1,os.path.getsize('./S1_buenosAires.csv')*8)
tasa_compresion_f_sin_extender_bo = tasa_compresion(clasificacion_bo,huffman_f_sin_extender_bo,1,os.path.getsize('./S2_bogota.csv')*8)
tasa_compresion_f_sin_extender_va = tasa_compresion(clasificacion_va,huffman_f_sin_extender_va,1,os.path.getsize('./S3_vancouver.csv')*8)

tasa_compresion_f_extendida_ba = tasa_compresion(clasificacion_ba,huffman_f_extendida_ba,2,os.path.getsize('./S1_buenosAires.csv')*8)
tasa_compresion_f_extendida_bo = tasa_compresion(clasificacion_bo,huffman_f_extendida_bo,2,os.path.getsize('./S2_bogota.csv')*8)
tasa_compresion_f_extendida_va = tasa_compresion(clasificacion_va,huffman_f_extendida_va,2,os.path.getsize('./S3_vancouver.csv')*8)

print("--------------------- Fuentes originales --------------------------")
print("Buenos Aires: ",f_sin_extender_ba)
print("Bogota: ",f_sin_extender_bo)
print("Vancouver: ",f_sin_extender_va)
print("")
print("--- Huffman para fuentes originales ---")
print("Buenos Aires: ",huffman_f_sin_extender_ba)
print("Bogota: ",huffman_f_sin_extender_bo)
print("Vancouver: ",huffman_f_sin_extender_va)
print("")
print("--- Longitud promedio para fuentes originales ---")
print("Buenos Aires: ",long_cod_f_sin_extender_ba)
print("Bogota: ",long_cod_f_sin_extender_bo)
print("Vancouver: ",long_cod_f_sin_extender_va)
print("")
print("--- Teorema de Shannon para fuentes originales ---" )
print("Buenos Aires: ",teorema_shannon(entropia_ba_sin_mem, long_cod_f_sin_extender_ba,0,1))
print("Bogota: ",teorema_shannon(entropia_bo_sin_mem, long_cod_f_sin_extender_bo,0,1))
print("Vancouver: ",teorema_shannon(entropia_va_sin_mem, long_cod_f_sin_extender_va,0,1))
print("")
print("--- Tasa de compresion para fuentes originales y longitud de codificacion ---")
print("Buenos Aires: ",tasa_compresion_f_sin_extender_ba)
print("Bogota: ",tasa_compresion_f_sin_extender_bo)
print("Vancouver: ",tasa_compresion_f_sin_extender_va)
print("")
print("--------------------- Fuentes extendidas --------------------------")
print("Buenos Aires: ",f_extendida_ba)
print("Bogota: ",f_extendida_bo)
print("Vancouver: ",f_extendida_va)
print("")
print("--- Huffman para fuentes extendidas ---")
print("Buenos Aires: ",huffman_f_extendida_ba)
print("Bogota: ",huffman_f_extendida_bo)
print("Vancouver: ",huffman_f_extendida_va)
print("")
print("--- Longitud promedio para fuentes extendidas ---")
print("Buenos Aires: ",long_cod_f_extendida_ba)
print("Bogota: ",long_cod_f_extendida_bo)
print("Vancouver: ",long_cod_f_extendida_va)
print("")
print("--- Teorema de Shannon para fuentes extendidas ---" )
print("Buenos Aires: ",teorema_shannon(entropia_ba_sin_mem, long_cod_f_extendida_ba,entropia_ba_cond,2))
print("Bogota: ",teorema_shannon(entropia_bo_sin_mem, long_cod_f_extendida_bo,entropia_bo_cond,2))
print("Vancouver: ",teorema_shannon(entropia_va_sin_mem, long_cod_f_extendida_va,entropia_va_cond,2))
print("")
print("--- Tasa de compresion para fuentes extendidas y longitud de codificacion ---")
print("Buenos Aires: ",tasa_compresion_f_extendida_ba)
print("Bogota: ",tasa_compresion_f_extendida_bo)
print("Vancouver: ",tasa_compresion_f_extendida_va)


## Ejercicio 3
Se quiere analizar las características de un **canal de información** que transmite las
temperaturas promedio registradas en una ciudad.
Para ello se tiene la señal **envíada S1** y la señal **recibida S4**

**a)** Crear una nueva señal T4
, a partir de S4 , teniendo en cuenta los rangos del ejercicio 2 y obtener la **matriz del canal** a partir de las señales T (entrada) y T4 (salida).

In [18]:
def calcular_matriz_canal(entrada,salida):
    cambio = defaultdict(Counter)
    for (estado_entrada, estado_salida) in zip(entrada, salida):
        cambio[estado_entrada][estado_salida] += 1

    matriz_canal = defaultdict(lambda: defaultdict(float))
    for estado_entrada in cambio:
      total_ocurrencias = sum(cambio[estado_entrada].values())
      for estado_salida in cambio[estado_entrada]:
        matriz_canal[estado_entrada][estado_salida] = cambio[estado_entrada][estado_salida] / total_ocurrencias

    return matriz_canal


clasificacion_ba_r = clasificar_temperatura(s4_buenos_aires_r)
matriz_canal_ba = calcular_matriz_canal(clasificacion_ba,clasificacion_ba_r)
print("--- Ejemplo 5 valores de temperatura (observar que varió de A a M) ---")
print("Buenos Aires entrada: ",clasificacion_ba[15:20])
print("Buenos Aires salida:  ", clasificacion_ba_r[15:20])

print("")
# Por ejemplo, las probabilidades de transicion de A hacia otro estado son
print("--- Matriz del canal ---")
pprint(dict(matriz_canal_ba))

--- Ejemplo 5 valores de temperatura (observar que varió de A a M) ---
Buenos Aires entrada:  ['A', 'M', 'A', 'A', 'A']
Buenos Aires salida:   ['A', 'M', 'M', 'A', 'A']

--- Matriz del canal ---
{'A': defaultdict(<class 'float'>,
                  {'A': 0.8474646716541978,
                   'M': 0.15253532834580216}),
 'B': defaultdict(<class 'float'>,
                  {'B': 0.8608870967741935,
                   'M': 0.13911290322580644}),
 'M': defaultdict(<class 'float'>,
                  {'A': 0.06154714850367024,
                   'B': 0.0815923207227555,
                   'M': 0.8568605307735743})}


**b)** Calcular el **ruido** y la **información mutua** del canal y analizar sus valores.

In [ ]:
def calcular_ruido(distr_fuente_entrada, matriz_canal): # H(Y/X)
  ruido = 0
  for entrada in distr_fuente_entrada:
    ruido_chiquito = 0
    probabilidad = distr_fuente_entrada[entrada]
    for salida in matriz_canal[entrada]:
      ruido_chiquito -= matriz_canal[entrada][salida] * np.log2(matriz_canal[entrada][salida])
    ruido_chiquito *= probabilidad
    ruido += ruido_chiquito

  return ruido

def calcular_informacion_mutua(probabilidad,matriz_prob_condicional,ruido): # I(X,Y) = H(Y) - ruido
  probabilidad_Y = defaultdict(float) # P(Y)
  for X in matriz_prob_condicional:
    for Y in matriz_prob_condicional[X]:
      probabilidad_Y[Y] += matriz_prob_condicional[X][Y] * probabilidad[X]

  #print(probabilidad_Y)

  H_Y = calcular_entropia(probabilidad_Y)
  #print(H_Y)

  return (H_Y - ruido)



In [ ]:

#ruido_prueba = calcular_ruido(distribucion_prueba,matriz_pasaje_prueba)
#print(ruido_prueba)
#print(calcular_informacion_mutua(probabilidad=distribucion_prueba,matriz_prob_condicional=matriz_pasaje_prueba,ruido=ruido_prueba))
ruido = calcular_ruido(distribucion_ba,matriz_canal_ba)
informacion_mutua = calcular_informacion_mutua(distribucion_ba,matriz_canal_ba,ruido)
print("--- Ruido del canal ---")
print(ruido)
print("")
print("--- Informacion mutua del canal ---")
print(informacion_mutua)

In [19]:
def distribucion_acumulada(distribucion):
  prob_acumulada = {}
  entrada = []
  prob = 0
  distribucion_ordenada = dict(sorted(distribucion.items(), key=lambda item: item[1]))

  for s in distribucion_ordenada:
    prob+=distribucion_ordenada[s]
    prob_acumulada[s]=prob
  return prob_acumulada

In [20]:
def matriz_acumulada(matriz_canal):
    prob_acumulada = {}
    prob = 0
    salida = []
    for s in matriz_canal:
      matriz_canal[s] = dict(sorted(matriz_canal[s].items(), key=lambda item: item[1]))

    for s_1 in matriz_canal:
      prob_acumulada_sim = {}
      prob=0
      for s_2 in matriz_canal[s_1]:
        prob+=matriz_canal[s_1][s_2]
        prob_acumulada_sim[s_2]=prob
      prob_acumulada[s_1] = prob_acumulada_sim
    return prob_acumulada

In [21]:
def generar_entrada(distribucion):
  prob_acumulada=distribucion_acumulada(distribucion)
  simbolo_alt = np.random.uniform()
  for s in prob_acumulada:
    if (simbolo_alt <= prob_acumulada[s]):
      return s


In [22]:
def generar_salida(matriz_canal, entrada):
  prob_acumulada = matriz_acumulada(matriz_canal)

  simbolo_alt = np.random.uniform()
  for s in prob_acumulada[entrada]:
    if (simbolo_alt <= prob_acumulada[entrada][s]):
        return s

In [ ]:
def media_recurrencia(simbolo,epsilon,MIN_ITERACIONES, distribucion, matriz_canal, n_simbolos_dist):
  retorno,media,t_actual = 0,0,0
  media_ant = -1
  s_e = generar_entrada(distribucion)

  exitos = 0
  iteraciones = 0
  prob_ant = -1
  prob_actual = 0
  cont=0
  flag_simbolo = False
  probabilidades=[0]
  iteraciones =[0]

  while (((prob_ant!=0)and(not abs(prob_ant-prob_actual)>epsilon)) or (t_actual < MIN_ITERACIONES)):
    t_actual += 1
    s_s = generar_salida(matriz_canal,s_e)
    #print("e: ",s_e, "- S: ",s_s)

    if((0<cont<=n_simbolos_dist)and(s_s == simbolo)):
      exitos+= 1

    if(s_s == simbolo):
      cont = 0
      flag_simbolo = True

    elif(flag_simbolo):
      cont+=1


    prob_ant=prob_actual
    prob_actual = exitos/t_actual

    s_e = generar_entrada(distribucion)

    probabilidades.append(prob_actual)
    iteraciones.append(t_actual)

  return probabilidades,iteraciones

In [ ]:
def grafico_convergencia(probabilidades,iteraciones):
  plt.plot(iteraciones, probabilidades, label='probabilidades')
  plt.show()

In [ ]:
prob,it=media_recurrencia('A',0.000001,6940,distribucion_ba,matriz_canal_ba,3)
print("Probabilidad de recurrencia: ", prob[-1])
grafico_convergencia(prob,it)
print("")

prob,it=media_recurrencia('A',0.000001,6940,distribucion_ba,matriz_canal_ba,2)
print("Probabilidad de recurrencia: ", prob[-1])
grafico_convergencia(prob,it)
print("")

prob,it=media_recurrencia('M',0.000001,6940,distribucion_ba,matriz_canal_ba,3)
print("Probabilidad de recurrencia: ", prob[-1])
grafico_convergencia(prob,it)
print("")

prob,it=media_recurrencia('B',0.000001,6940,distribucion_ba,matriz_canal_ba,3)
print("Probabilidad de recurrencia: ", prob[-1])
grafico_convergencia(prob,it)
print("")